# U2-08 · Redes generativas antagónicas y atención

*Programación de Redes Neuronales - Machine Learning (IFCD48) · Unidad 2, apartados 2.10 a 2.12*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con el botón ▶). Activa la GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución*) para la GAN de dígitos: sin ella tarda unos 20 minutos.

> Datos: MNIST, de Y. LeCun, C. Cortes y C. J. C. Burges (CC BY-SA 3.0). Keras lo descarga en la primera ejecución.

## 1. Una GAN en una dimensión

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

def reales(n, rng):                  # dos picos: en -2 y en +2
    centro = rng.choice([-2.0, 2.0], n)
    return (centro + rng.normal(0, 0.5, n)).astype("float32")[:, None]

def red(salida):
    return keras.Sequential([keras.Input(shape=(1,)),
                             layers.Dense(32, "relu"),
                             layers.Dense(32, "relu"),
                             layers.Dense(1, activation=salida)])

keras.utils.set_random_seed(0)
rng = np.random.default_rng(0)
G, D = red(None), red(None)          # generador y discriminador
opt_g = keras.optimizers.Adam(1e-3)
opt_d = keras.optimizers.Adam(1e-3)
bce = keras.losses.BinaryCrossentropy(from_logits=True)

@tf.function
def paso(x_real):
    z = tf.random.normal((len(x_real), 1))
    with tf.GradientTape() as cd, tf.GradientTape() as cg:
        x_falso = G(z)
        d_real, d_falso = D(x_real), D(x_falso)
        perd_d = (bce(tf.ones_like(d_real), d_real)       # 1 = real
                  + bce(tf.zeros_like(d_falso), d_falso)) # 0 = falso
        perd_g = bce(tf.ones_like(d_falso), d_falso)      # engañar
    opt_d.apply_gradients(zip(cd.gradient(perd_d, D.weights),
                              D.weights))
    opt_g.apply_gradients(zip(cg.gradient(perd_g, G.weights),
                              G.weights))
    return perd_d, perd_g

for t in range(1, 3001):
    pd_, pg = paso(reales(256, rng))
    if t in (1, 500, 1000, 3000):
        m = G(tf.random.normal((5000, 1))).numpy()[:, 0]
        print(f"paso {t:4d}: pérdida D {float(pd_):.3f}, "
              f"G {float(pg):.3f} | izquierda {np.mean(m < 0):.2f}, "
              f"desv. {m.std():.2f}")

x = reales(5000, rng)[:, 0]
plt.hist(x, 60, density=True, alpha=0.5, label="datos reales")
plt.hist(m, 60, density=True, alpha=0.5, label="generados")
plt.legend(); plt.show()  # plt.savefig("gan1d.pdf")
for nombre, v in [("reales", x), ("generados", m)]:
    print(f"{nombre}: desv. dentro de cada pico "
          f"{v[v < 0].std():.2f} y {v[v > 0].std():.2f}")
print(f"equilibrio teórico de D: 2 ln 2 = {2 * np.log(2):.3f}")

**Prueba.** Cambia los datos reales a tres picos (en −3, 0 y 3). ¿Los cubre todos el generador?

## 2. Una GAN que genera dígitos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

(x, y), _ = keras.datasets.mnist.load_data()
x = (x.reshape(-1, 784).astype("float32") - 127.5) / 127.5  # [-1, 1]

keras.utils.set_random_seed(0)
G = keras.Sequential([keras.Input(shape=(64,)),
                      layers.Dense(256), layers.LeakyReLU(0.2),
                      layers.Dense(512), layers.LeakyReLU(0.2),
                      layers.Dense(784, activation="tanh")])
D = keras.Sequential([keras.Input(shape=(784,)),
                      layers.Dense(512), layers.LeakyReLU(0.2),
                      layers.Dense(256), layers.LeakyReLU(0.2),
                      layers.Dense(1)])
opt_g = keras.optimizers.Adam(2e-4, beta_1=0.5)
opt_d = keras.optimizers.Adam(2e-4, beta_1=0.5)
bce = keras.losses.BinaryCrossentropy(from_logits=True)

@tf.function
def paso(x_real):
    z = tf.random.normal((tf.shape(x_real)[0], 64))
    with tf.GradientTape() as cd, tf.GradientTape() as cg:
        d_real, d_falso = D(x_real), D(G(z))
        perd_d = (bce(tf.ones_like(d_real), d_real)
                  + bce(tf.zeros_like(d_falso), d_falso))
        perd_g = bce(tf.ones_like(d_falso), d_falso)
    opt_d.apply_gradients(zip(cd.gradient(perd_d, D.weights),
                              D.weights))
    opt_g.apply_gradients(zip(cg.gradient(perd_g, G.weights),
                              G.weights))
    return perd_d, perd_g

z_fijo = tf.random.normal((8, 64))       # los mismos 8 puntos siempre
lotes = tf.data.Dataset.from_tensor_slices(x)
lotes = lotes.shuffle(60_000).batch(128)
filas = []
for epoca in range(1, 21):
    for lote in lotes:
        perd_d, perd_g = paso(lote)
    if epoca in (1, 5, 20):
        filas.append(G(z_fijo).numpy().reshape(8, 28, 28))
        print(f"época {epoca:2d}: pérdida D {float(perd_d):.3f}, "
              f"G {float(perd_g):.3f}")

# ¿qué dígitos genera? Los clasifica una red entrenada aparte
juez = keras.Sequential([keras.Input(shape=(784,)),
                         layers.Dense(128, activation="relu"),
                         layers.Dense(10, activation="softmax")])
juez.compile("adam", "sparse_categorical_crossentropy")
juez.fit(x, y, epochs=3, verbose=0)
p = juez.predict(G(tf.random.normal((5000, 64))), verbose=0)
print("dígitos generados:", np.bincount(p.argmax(1), minlength=10))
print(f"con confianza > 0.9: {np.mean(p.max(1) > 0.9):.2f}")

fig, ejes = plt.subplots(3, 8, figsize=(4.4, 1.8))
for f, (fila, ep) in enumerate(zip(filas, [1, 5, 20])):
    for c in range(8):
        ejes[f, c].imshow(fila[c]); ejes[f, c].axis("off")
    ejes[f, 0].set_title(f"época {ep}", loc="left", fontsize=6)
plt.show()  # fig.savefig("gan_mnist.pdf")

**Prueba.** Entrena 40 épocas. ¿Se reparten mejor los dígitos generados? ¿Sube la proporción con confianza > 0.9?

## 3. Pedir un dígito: la GAN condicional

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

(x, y), _ = keras.datasets.mnist.load_data()
x = (x.reshape(-1, 784).astype("float32") - 127.5) / 127.5
y1 = keras.utils.to_categorical(y, 10)       # etiqueta en one-hot

keras.utils.set_random_seed(0)
z_e, c_e = keras.Input(shape=(64,)), keras.Input(shape=(10,))
h = layers.Concatenate()([z_e, c_e])         # ruido + dígito pedido
for n in (256, 512):
    h = layers.LeakyReLU(0.2)(layers.Dense(n)(h))
G = keras.Model([z_e, c_e], layers.Dense(784, "tanh")(h))
x_e, c_d = keras.Input(shape=(784,)), keras.Input(shape=(10,))
h = layers.Concatenate()([x_e, c_d])         # imagen + dígito dicho
for n in (512, 256):
    h = layers.LeakyReLU(0.2)(layers.Dense(n)(h))
D = keras.Model([x_e, c_d], layers.Dense(1)(h))
opt_g = keras.optimizers.Adam(2e-4, beta_1=0.5)
opt_d = keras.optimizers.Adam(2e-4, beta_1=0.5)
bce = keras.losses.BinaryCrossentropy(from_logits=True)

@tf.function
def paso(xr, c):
    z = tf.random.normal((tf.shape(xr)[0], 64))
    with tf.GradientTape() as cd, tf.GradientTape() as cg:
        dr, df = D([xr, c]), D([G([z, c]), c])
        pd_ = bce(tf.ones_like(dr), dr) + bce(tf.zeros_like(df), df)
        pg = bce(tf.ones_like(df), df)
    opt_d.apply_gradients(zip(cd.gradient(pd_, D.weights),
                              D.weights))
    opt_g.apply_gradients(zip(cg.gradient(pg, G.weights),
                              G.weights))

lotes = (tf.data.Dataset.from_tensor_slices((x, y1))
         .shuffle(60_000, seed=0).batch(128))
for epoca in range(20):
    for xr, c in lotes:
        paso(xr, c)

juez = keras.Sequential([keras.Input(shape=(784,)),   # otra red
                         layers.Dense(128, activation="relu"),
                         layers.Dense(10, activation="softmax")])
juez.compile("adam", "sparse_categorical_crossentropy")
juez.fit(x, y, epochs=3, verbose=0)
pedido = np.repeat(np.arange(10), 500)
img = G.predict([np.random.default_rng(0).normal(size=(5000, 64)),
                 keras.utils.to_categorical(pedido, 10)], verbose=0)
leido = juez.predict(img, verbose=0).argmax(1)
print(f"el juez lee el dígito pedido en el "
      f"{np.mean(leido == pedido):.1%} de 5000 imágenes")
por_digito = [np.mean(leido[pedido == d] == d) for d in range(10)]
print("acierto por dígito:", np.round(por_digito, 2))

fig, ejes = plt.subplots(3, 10, figsize=(4.4, 1.4))
for d in range(10):
    for f in range(3):
        ejes[f, d].imshow(img[d * 500 + f].reshape(28, 28))
        ejes[f, d].axis("off")
plt.show()  # fig.savefig("condicional.pdf")

## 4. El colapso de modos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

ang = np.arange(8) * np.pi / 4
MODOS = np.stack([2 * np.cos(ang), 2 * np.sin(ang)], 1)  # 8 picos
def reales(n, rng):
    return (MODOS[rng.integers(8, size=n)]
            + rng.normal(0, 0.05, (n, 2))).astype("float32")

def red(entrada, salida):
    return keras.Sequential([keras.Input(shape=(entrada,)),
                             layers.Dense(128, "relu"),
                             layers.Dense(128, "relu"),
                             layers.Dense(salida)])

def cubiertos(m):                       # picos con al menos 2 %
    d = np.linalg.norm(m[:, None] - MODOS[None], axis=2)
    cerca = d.min(1) < 0.3
    return np.sum(np.bincount(d.argmin(1)[cerca], minlength=8)
                  > 0.02 * len(m)), cerca.mean()

keras.utils.set_random_seed(0); rng = np.random.default_rng(0)
G, D = red(2, 2), red(2, 1)
opt_g = keras.optimizers.Adam(1e-3)
opt_d = keras.optimizers.Adam(1e-3)
bce = keras.losses.BinaryCrossentropy(from_logits=True)

@tf.function
def paso(xr):
    z = tf.random.normal((tf.shape(xr)[0], 2))
    with tf.GradientTape() as cd, tf.GradientTape() as cg:
        dr, df = D(xr), D(G(z))
        pd_ = bce(tf.ones_like(dr), dr) + bce(tf.zeros_like(df), df)
        pg = bce(tf.ones_like(df), df)
    opt_d.apply_gradients(zip(cd.gradient(pd_, D.weights),
                              D.weights))
    opt_g.apply_gradients(zip(cg.gradient(pg, G.weights),
                              G.weights))

fig, ejes = plt.subplots(1, 4, figsize=(4.4, 1.2))
for t in range(1, 6001):
    paso(reales(256, rng))
    if t in (500, 1500, 3000, 6000):
        m = G(tf.random.normal((2000, 2))).numpy()
        n, frac = cubiertos(m)
        print(f"paso {t:4d}: picos cubiertos {n} de 8, "
              f"puntos cerca de un pico {frac:.0%}")
        e = ejes[(500, 1500, 3000, 6000).index(t)]
        e.scatter(*MODOS.T, s=8, c="0.7"); e.scatter(*m.T, s=0.5)
        e.set_title(f"paso {t}", loc="center"); e.axis("off")
        e.set_xlim(-3, 3); e.set_ylim(-3, 3)
plt.show()  # fig.savefig("modos.pdf")

## 5. La atención, a mano

In [ ]:
import numpy as np

def atencion(Q, K, V):
    """Atención de producto escalar: cada consulta reparte su peso
    entre las claves parecidas y devuelve la media de sus valores."""
    s = Q @ K.T / np.sqrt(K.shape[1])          # parecido
    w = np.exp(s - s.max(1, keepdims=True))
    w /= w.sum(1, keepdims=True)                # softmax por fila
    return w @ V, w

K = np.array([[1., 0, 0, 0], [0, 1, 0, 0],     # 4 posiciones
              [0, 0, 1, 0], [0, 0, 0, 1]]) * 3  # de memoria
V = np.array([[10.], [20.], [30.], [40.]])      # lo que guardan
Q = np.array([[0, 0, 3., 0],                    # busca la 3.ª
              [0, 3., 3., 0]])                  # busca la 2.ª y 3.ª
salida, w = atencion(Q, K, V)
print("pesos:\n", w.round(3))
print("salida:", salida.ravel().round(2))

## 6. Atención frente a LSTM en el problema de la suma

La primera celda guarda como archivo el código del apartado 2.8, que la segunda importa. Tarda unos minutos.

In [ ]:
%%writefile v_memoria.py
import numpy as np
import keras
from keras import layers

def sumas(n, largo, semilla):
    """Problema de la suma: hay que sumar los dos valores marcados,
    uno en la primera mitad de la secuencia y otro en la segunda."""
    rng = np.random.default_rng(semilla)
    v = rng.uniform(0, 1, (n, largo))
    m = np.zeros((n, largo))
    m[np.arange(n), rng.integers(0, largo // 2, n)] = 1
    m[np.arange(n), rng.integers(largo // 2, largo, n)] = 1
    x = np.stack([v, m], axis=-1).astype("float32")
    return x, (v * m).sum(1).astype("float32")

if __name__ == "__main__":          # solo al ejecutar este archivo
    for largo in [10, 50, 100]:
        x, y = sumas(10_000, largo, 0)
        xp, yp = sumas(1000, largo, 1)
        for tipo in [layers.SimpleRNN, layers.LSTM]:
            keras.utils.set_random_seed(0)
            red = keras.Sequential([keras.Input(shape=(largo, 2)),
                                    tipo(32), layers.Dense(1)])
            red.compile("adam", "mse")
            red.fit(x, y, epochs=20, batch_size=64, verbose=0)
            ecm = red.evaluate(xp, yp, verbose=0)
            print(f"{largo:3d} pasos, {tipo.__name__:<9}: error "
                  f"{ecm:.4f}")
    print(f"sin memoria (responder siempre 1): "
          f"{np.mean((yp - 1) ** 2):.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers, ops
from v_memoria import sumas                # el problema de 2.8

largo = 150
x, y = sumas(10_000, largo, 0)
xp, yp = sumas(1000, largo, 1)

def con_atencion():
    entrada = keras.Input(shape=(largo, 2))
    h = layers.Dense(32, activation="tanh")(entrada)   # cada paso
    puntos = layers.Dense(1)(h)                        # relevancia
    pesos = layers.Softmax(axis=1, name="pesos")(puntos)  # suman 1
    contexto = ops.sum(pesos * h, axis=1)              # media pond.
    oculta = layers.Dense(32, activation="relu")(contexto)
    return keras.Model(entrada, layers.Dense(1)(oculta))

def con_lstm():
    return keras.Sequential([keras.Input(shape=(largo, 2)),
                             layers.LSTM(32), layers.Dense(1)])

for nombre, construir in [("atención", con_atencion),
                          ("LSTM", con_lstm)]:
    keras.utils.set_random_seed(0)
    m = construir()
    m.compile("adam", "mse")
    m.fit(x, y, epochs=20, batch_size=64, verbose=0)
    print(f"{largo} pasos, {nombre:<8}: error "
          f"{m.evaluate(xp, yp, verbose=0):.2e}")
    if nombre == "atención":
        ver = keras.Model(m.inputs, m.get_layer("pesos").output)
        w = ver.predict(xp[:1], verbose=0)[0, :, 0]

marcas = np.flatnonzero(xp[0, :, 1])
print("posiciones marcadas:", marcas,
      f"| peso total sobre ellas: {w[marcas].sum():.3f}")
plt.plot(w); plt.plot(marcas, w[marcas], "o")
plt.xlabel("paso"); plt.ylabel("peso de atención")
plt.show()  # plt.savefig("atencion.pdf")

## 7. Atención que no mira el futuro

In [ ]:
import numpy as np

def atencion(Q, K, V, causal=False):
    s = Q @ K.T / np.sqrt(K.shape[1])        # parecidos
    if causal:                                # no mirar el futuro
        s = np.where(np.tril(np.ones_like(s)) == 1, s, -np.inf)
    w = np.exp(s - s.max(1, keepdims=True))
    w = w / w.sum(1, keepdims=True)
    return w @ V, w

rng = np.random.default_rng(0)
X = rng.normal(size=(5, 4))                   # 5 palabras, 4 números
for causal in (False, True):
    sal, w = atencion(X, X, X, causal)
    X2 = X.copy(); X2[4] += 10                # cambia la última
    sal2, _ = atencion(X2, X2, X2, causal)
    cambian = np.abs(sal2 - sal).max(1) > 1e-9
    print(f"causal {causal!s:<5}: pesos de la posición 2 ->",
          w[2].round(2), "| salidas que cambian:",
          np.flatnonzero(cambian))

## 8. Codificar la posición

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def posiciones(n, d):               # codificación sinusoidal
    pos = np.arange(n)[:, None]
    i = np.arange(d // 2)[None]
    ang = pos / 10_000 ** (2 * i / d)
    P = np.zeros((n, d))
    P[:, 0::2], P[:, 1::2] = np.sin(ang), np.cos(ang)
    return P

P = posiciones(50, 64)
sim = P @ P.T / 32                     # parecido entre posiciones
print("parecido de la posición 10 con 9, 11, 20 y 40:",
      sim[10, [9, 11, 20, 40]].round(2))
print("¿depende solo de la distancia?",
      np.isclose(sim[10, 15], sim[30, 35]))
fig, (a, b) = plt.subplots(1, 2, figsize=(4.4, 1.8),
                           layout="constrained")
a.imshow(P, cmap="RdBu", aspect="auto")
a.set_xlabel("dimensión"); a.set_ylabel("posición")
b.plot(sim[10]); b.set_xlabel("posición")
b.set_title("parecido con la 10")
plt.show()  # fig.savefig("posicion.pdf")

## 9. Un Transformer pequeño

In [ ]:
import time
import keras
from keras import layers

(x, y), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x / 255.0, xp / 255.0           # 28 filas de 28 píxeles

class Posicion(layers.Layer):            # suma un vector por fila
    def build(self, forma):
        self.p = self.add_weight(shape=forma[1:], name="p",
                                 initializer="random_normal")
    def call(self, h):
        return h + self.p

keras.utils.set_random_seed(0)
e = keras.Input(shape=(28, 28))
h = Posicion()(layers.Dense(64)(e))           # cada fila: 64 números
a = layers.MultiHeadAttention(num_heads=4, key_dim=16)(h, h)
h = layers.LayerNormalization()(h + a)        # atención + residuo
f = layers.Dense(128, "relu")(h)
h = layers.LayerNormalization()(h + layers.Dense(64)(f))
s = layers.Dense(10, "softmax")(layers.GlobalAveragePooling1D()(h))
red = keras.Model(e, s)
red.compile("adam", "sparse_categorical_crossentropy",
            metrics=["accuracy"])
t = time.perf_counter()
red.fit(x[:50_000], y[:50_000], epochs=15, batch_size=128,
        validation_data=(x[50_000:], y[50_000:]), verbose=0)
print(f"{red.count_params():,} parámetros, "
      f"{time.perf_counter() - t:.0f} s, prueba "
      f"{red.evaluate(xp, yp, verbose=0)[1]:.4f}")

## 10. Mirar dentro de la atención

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

(x, y), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x / 255.0, xp / 255.0

keras.utils.set_random_seed(0)              # una sola cabeza
e = keras.Input(shape=(28, 28))
h = layers.Dense(64)(e)
h = h + layers.Embedding(28, 64)(np.arange(28))   # posición de fila
atencion = layers.MultiHeadAttention(num_heads=1, key_dim=64)
a, pesos = atencion(h, h, return_attention_scores=True)
h = layers.LayerNormalization()(h + a)
s = layers.GlobalAveragePooling1D()(h)
red = keras.Model(e, layers.Dense(10, "softmax")(s))
red.compile("adam", "sparse_categorical_crossentropy",
            metrics=["accuracy"])
red.fit(x, y, epochs=10, batch_size=128, verbose=0)
print(f"prueba {red.evaluate(xp, yp, verbose=0)[1]:.4f}")

ver = keras.Model(e, pesos)                  # extrae los pesos
P = ver.predict(xp[:1000], verbose=0)[:, 0]  # (1000, 28, 28)
recibe = P.mean(axis=(0, 1))                 # atención recibida
print("filas más atendidas:", np.argsort(recibe)[::-1][:5])
print("atención a las filas 0-3 y 24-27:",
      recibe[:4].sum().round(3), recibe[24:].sum().round(3))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(4.4, 2.0),
                             layout="constrained")
a1.imshow(xp[0]); a1.axis("off")
a2.imshow(P[0], cmap="Blues")
a2.set_xlabel("fila a la que atiende")
a2.set_ylabel("fila que pregunta")
plt.show()  # fig.savefig("mapa_atencion.pdf")

## 11. Proyecto: reducir el modelo y probar su robustez (2.12)

La primera celda descarga la red convolucional del proyecto de la web del libro.

In [ ]:
import urllib.request              # la red del proyecto (apartado 2.12)
urllib.request.urlretrieve("https://especialidades-formativas.github.io/ifcd48/recursos/u2/app/mnist_cnn.keras", "mnist_cnn.keras")

In [ ]:
import json, time
import numpy as np
import keras

red = keras.models.load_model("mnist_cnn.keras")   # la del proyecto
(_, _), (xp, yp) = keras.datasets.mnist.load_data()
xp = xp[..., None] / 255.0
base = red.predict(xp, verbose=0).argmax(1)
print(f"original: acierto {np.mean(base == yp):.4f}")

def prueba(pesos):                     # acierto con otros pesos
    copia = keras.models.clone_model(red)
    copia.set_weights([w.astype(np.float32) for w in pesos])
    p = copia.predict(xp, verbose=0).argmax(1)
    return np.mean(p == yp), np.sum(p != base)

for dec in (6, 3, 2):                  # JSON con menos decimales
    pesos = [np.round(w.astype(float), dec)
             for w in red.get_weights()]
    mb = len(json.dumps([w.tolist() for w in pesos])) / 1e6
    print(f"JSON con {dec} decimales: {mb:.2f} MB, acierto %.4f, "
          f"distintas %d" % prueba(pesos))
for tipo in (np.float32, np.float16):  # binario de 32 o 16 bits
    pesos = [w.astype(tipo) for w in red.get_weights()]
    mb = sum(w.nbytes for w in pesos) / 1e6
    print(f"binario {tipo.__name__}: {mb:.2f} MB, acierto %.4f, "
          f"distintas %d" % prueba(pesos))

t = time.perf_counter()
for i in range(100):
    red(xp[i:i + 1], training=False)          # de una en una
print(f"latencia media por imagen: "
      f"{(time.perf_counter() - t) * 10:.1f} ms")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
import keras

red = keras.models.load_model("mnist_cnn.keras")   # la del proyecto
(_, _), (xp, yp) = keras.datasets.mnist.load_data()
xp, yp = xp[:2000] / 255.0, yp[:2000]

def acierto(imgs):
    return np.mean(red.predict(imgs[..., None], verbose=0)
                   .argmax(1) == yp)

desp = [0, 2, 4, 6]
giro = [0, 15, 30, 45]
a_d = [acierto(np.stack([ndimage.shift(i, (0, d)) for i in xp]))
       for d in desp]
a_g = [acierto(np.stack([ndimage.rotate(i, g, reshape=False)
                         for i in xp])) for g in giro]
print("desplazamiento (px):", desp, "->", np.round(a_d, 3))
print("giro (grados):      ", giro, "->", np.round(a_g, 3))
borrosa = np.stack([ndimage.gaussian_filter(i, 1.5) for i in xp])
print("desenfocada:", round(acierto(borrosa), 3),
      "| colores invertidos:", round(acierto(1 - xp), 3))

fig, (a, b) = plt.subplots(1, 2, figsize=(4.4, 1.6), sharey=True,
                           layout="constrained")
a.plot(desp, a_d, "o-"); a.set_xlabel("desplazamiento (píxeles)")
b.plot(giro, a_g, "o-"); b.set_xlabel("giro (grados)")
a.set_ylabel("acierto"); plt.show()  # fig.savefig("robustez.pdf")

In [ ]:
import numpy as np
import keras
from scipy import ndimage

red = keras.models.load_model("mnist_cnn.keras")   # la del proyecto
(_, _), (xp, yp) = keras.datasets.mnist.load_data()
(_, _), (fp, _) = keras.datasets.fashion_mnist.load_data()
conf = lambda imgs: red.predict(imgs[..., None] / 255.0,
                                verbose=0).max(1)
girados = np.stack([ndimage.rotate(i, 45, reshape=False)
                    for i in xp[:2000]])
lotes = {"dígitos de prueba": xp[:2000],
         "dígitos girados 45°": girados,
         "prendas (otra cosa)": fp[:2000]}
base = conf(xp[2000:4000])               # referencia: datos normales
umbral = np.quantile(base, 0.05)
print(f"umbral: el 5 % de los dígitos normales baja de {umbral:.3f}")
for nombre, imgs in lotes.items():
    c = conf(imgs)
    print(f"{nombre:<20} confianza media {c.mean():.3f}, por debajo "
          f"del umbral {np.mean(c < umbral):.0%}")

**Ejercicio.** Prueba la red con atención con secuencias de 500 pasos. ¿Sigue encontrando las dos posiciones marcadas?

> Goodfellow, I. et al. (2020). Generative adversarial networks. *Communications of the ACM*, 63(11), 139–144. https://doi.org/10.1145/3422622
>
> Vaswani, A. et al. (2017). Attention is all you need. arXiv:1706.03762. https://doi.org/10.48550/arXiv.1706.03762